# Enhanced DeFooocus Colab Launcher

This notebook is designed to reduce repetitive setup work in Colab by adding GPU checks, optional Google Drive persistence, idempotent repository bootstrap, and a simplified launch flow.

In [ ]:
#@title Runtime configuration
PROJECT_REPO_URL = "https://github.com/henryzengyh/DeFooocus.git" #@param {type:"string"}
PROJECT_REPO_REF = "main" #@param {type:"string"}
ENABLE_DRIVE = True #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/AI/enhanced_defooocus" #@param {type:"string"}
RUNTIME_ROOT = "/content/runtime/enhanced_defooocus" #@param {type:"string"}
AUTO_UPDATE_REPO = True #@param {type:"boolean"}
SHARE_GRADIO = True #@param {type:"boolean"}
MOBILE_MODE = True #@param {type:"boolean"}

import os
from pathlib import Path

os.environ["PROJECT_REPO_URL"] = PROJECT_REPO_URL
os.environ["PROJECT_REPO_REF"] = PROJECT_REPO_REF
os.environ["PERSIST_TO_DRIVE"] = "1" if ENABLE_DRIVE else "0"
os.environ["PERSIST_ROOT"] = DRIVE_ROOT
os.environ["RUNTIME_ROOT"] = RUNTIME_ROOT
os.environ["AUTO_UPDATE_REPO"] = "1" if AUTO_UPDATE_REPO else "0"
os.environ["SHARE_GRADIO"] = "1" if SHARE_GRADIO else "0"
os.environ["MOBILE_MODE"] = "1" if MOBILE_MODE else "0"

Path(RUNTIME_ROOT).mkdir(parents=True, exist_ok=True)
Path("/content/runtime/helper_repo").mkdir(parents=True, exist_ok=True)
print("Configuration loaded")

In [ ]:
# Mount Google Drive when persistence is enabled.
if ENABLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print(f"Drive mounted at {DRIVE_ROOT}")
else:
    print("Drive persistence disabled")

In [ ]:
# Materialize helper scripts so the notebook can self-bootstrap even when opened directly from Drive.
from pathlib import Path

helper_root = Path('/content/runtime/helper_repo/enhanced_defooocus_colab')
scripts_dir = helper_root / 'scripts'
scripts_dir.mkdir(parents=True, exist_ok=True)

bootstrap_sh = r'''#!/usr/bin/env bash
set -euo pipefail

PROJECT_REPO_URL="${PROJECT_REPO_URL:-https://github.com/henryzengyh/DeFooocus.git}"
PROJECT_REPO_REF="${PROJECT_REPO_REF:-main}"
RUNTIME_ROOT="${RUNTIME_ROOT:-/content/runtime/enhanced_defooocus}"
REPO_DIR="${REPO_DIR:-$RUNTIME_ROOT/repo}"
PERSIST_ROOT="${PERSIST_ROOT:-/content/drive/MyDrive/AI/enhanced_defooocus}"
PERSIST_TO_DRIVE="${PERSIST_TO_DRIVE:-1}"
AUTO_UPDATE_REPO="${AUTO_UPDATE_REPO:-1}"
PYTHON_BIN="${PYTHON_BIN:-python3}"
PIP_BIN="${PIP_BIN:-pip3}"

log() {
  printf '\n[%s] %s\n' "$(date '+%H:%M:%S')" "$1"
}

ensure_gpu() {
  if ! command -v nvidia-smi >/dev/null 2>&1; then
    echo "No GPU runtime detected. Switch Colab runtime to GPU first." >&2
    exit 1
  fi
  nvidia-smi || true
}

clone_or_update_repo() {
  mkdir -p "$RUNTIME_ROOT"
  if [[ ! -d "$REPO_DIR/.git" ]]; then
    log "Cloning project repository"
    git clone --depth 1 --branch "$PROJECT_REPO_REF" "$PROJECT_REPO_URL" "$REPO_DIR"
    return
  fi

  if [[ "$AUTO_UPDATE_REPO" != "1" ]]; then
    log "Repository already present; skipping update"
    return
  fi

  log "Updating existing repository"
  git -C "$REPO_DIR" fetch --depth 1 origin "$PROJECT_REPO_REF"
  git -C "$REPO_DIR" checkout -q "$PROJECT_REPO_REF"
  git -C "$REPO_DIR" reset --hard "origin/$PROJECT_REPO_REF"
}

prepare_persistent_dirs() {
  if [[ "$PERSIST_TO_DRIVE" != "1" ]]; then
    log "Drive persistence disabled"
    return
  fi

  log "Preparing persistent directories"
  mkdir -p \
    "$PERSIST_ROOT/models/checkpoints" \
    "$PERSIST_ROOT/models/loras" \
    "$PERSIST_ROOT/models/embeddings" \
    "$PERSIST_ROOT/outputs"

  mkdir -p "$REPO_DIR/models"
  rm -rf "$REPO_DIR/models/checkpoints" "$REPO_DIR/models/loras" "$REPO_DIR/models/embeddings" "$REPO_DIR/outputs"
  ln -sfn "$PERSIST_ROOT/models/checkpoints" "$REPO_DIR/models/checkpoints"
  ln -sfn "$PERSIST_ROOT/models/loras" "$REPO_DIR/models/loras"
  ln -sfn "$PERSIST_ROOT/models/embeddings" "$REPO_DIR/models/embeddings"
  ln -sfn "$PERSIST_ROOT/outputs" "$REPO_DIR/outputs"
}

install_dependencies() {
  log "Installing base packages"
  apt-get -qq update
  apt-get -qq install -y git aria2 >/dev/null

  log "Upgrading pip tooling"
  "$PYTHON_BIN" -m pip install --upgrade pip setuptools wheel

  if [[ -f "$REPO_DIR/requirements_versions.txt" ]]; then
    log "Installing requirements_versions.txt"
    "$PIP_BIN" install -r "$REPO_DIR/requirements_versions.txt"
  elif [[ -f "$REPO_DIR/requirements.txt" ]]; then
    log "Installing requirements.txt"
    "$PIP_BIN" install -r "$REPO_DIR/requirements.txt"
  else
    echo "No requirements file found in repository." >&2
    exit 1
  fi
}

apply_compat_fixes() {
  log "Applying compatibility fixes for Colab runtime"
  "$PIP_BIN" uninstall -y cupy cupy-cuda11x cupy-cuda12x >/dev/null 2>&1 || true
  "$PIP_BIN" install -q "numpy==1.26.4"
}

main() {
  ensure_gpu
  clone_or_update_repo
  prepare_persistent_dirs
  install_dependencies
  apply_compat_fixes
  log "Bootstrap completed"
}

main "$@"
'''

launch_sh = r'''#!/usr/bin/env bash
set -euo pipefail

RUNTIME_ROOT="${RUNTIME_ROOT:-/content/runtime/enhanced_defooocus}"
REPO_DIR="${REPO_DIR:-$RUNTIME_ROOT/repo}"
SHARE_GRADIO="${SHARE_GRADIO:-1}"
PYTHON_BIN="${PYTHON_BIN:-python3}"
MOBILE_MODE="${MOBILE_MODE:-1}"

log() {
  printf '\n[%s] %s\n' "$(date '+%H:%M:%S')" "$1"
}

build_launch_args() {
  local args=()
  if [[ "$SHARE_GRADIO" == "1" ]]; then
    args+=("--share")
  fi
  printf '%s\n' "${args[@]}"
}

main() {
  cd "$REPO_DIR"
  if [[ "$MOBILE_MODE" == "1" ]]; then
    log "Mobile mode enabled: prefer lighter presets inside the UI after startup"
  fi
  mapfile -t launch_args < <(build_launch_args)
  if [[ -f "entry_with_update.py" ]]; then
    log "Launching entry_with_update.py"
    "$PYTHON_BIN" entry_with_update.py "${launch_args[@]}"
    exit 0
  fi
  if [[ -f "launch.py" ]]; then
    log "Launching launch.py"
    "$PYTHON_BIN" launch.py "${launch_args[@]}"
    exit 0
  fi
  echo "No known launch entrypoint found (entry_with_update.py or launch.py)." >&2
  exit 1
}

main "$@"
'''

(scripts_dir / 'bootstrap.sh').write_text(bootstrap_sh)
(scripts_dir / 'launch_webui.sh').write_text(launch_sh)
!chmod +x /content/runtime/helper_repo/enhanced_defooocus_colab/scripts/bootstrap.sh
!chmod +x /content/runtime/helper_repo/enhanced_defooocus_colab/scripts/launch_webui.sh
print('Helper scripts materialized')

In [ ]:
# Bootstrap repository and dependencies.
!bash /content/runtime/helper_repo/enhanced_defooocus_colab/scripts/bootstrap.sh

In [ ]:
# Launch the WebUI. When share is enabled, Colab will print a public Gradio URL.
!bash /content/runtime/helper_repo/enhanced_defooocus_colab/scripts/launch_webui.sh